In [2]:
"""Train a per-lender approval-likelihood model and save it to models/."""

import warnings
import sklearn
import pandas as pd
from tqdm import tqdm
import joblib
from loan_approval.config import MODEL_DIR
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from loan_approval.config import DATA_PROCESSED_DIR
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Expected: rare categories (e.g. an aus-3 code outside the -1 sentinel) can be
# absent from a given CV fold's training split -- handle_unknown="ignore"
# already handles it correctly, this just quiets the noise.
warnings.filterwarnings("ignore", message="Found unknown categories")


In [7]:

#Import data
df = pd.read_csv(DATA_PROCESSED_DIR / "hmda_features.csv")
df.drop(columns=["Unnamed: 0"], inplace=True)

# Prepare training data
target_col = "approved"
X = df.drop(columns=[target_col])
y = df[target_col]

categorical_cols = [
    "lei", "loan_type", "loan_purpose", "occupancy_type",
    "applicant_credit_score_type", "co-applicant_credit_score_type",
    "interest_only_payment", "negative_amortization", "balloon_payment",
    "aus-1", "aus-2", "aus-3",
]
numerical_cols = [col for col in X.columns if col not in categorical_cols]

def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("ohe", OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore"), categorical_cols),
            ("scaler", StandardScaler(), numerical_cols),
        ]
    )

,lei,loan_type,loan_purpose,loan_amount,loan_term,negative_amortization,interest_only_payment,balloon_payment,occupancy_type,income,debt_to_income_ratio,applicant_credit_score_type,co-applicant_credit_score_type,aus-1,aus-2,aus-3,approved,num_aus_used
0,B4TYDEB6GKMZO031MB27,1,1,45000.0,240.0,2,2,2,3,42.0,37.0,3,10,2,-1,-1,1,1
1,B4TYDEB6GKMZO031MB27,2,1,215000.0,360.0,2,2,2,1,129.0,25.0,9,3,3,-1,-1,1,1
2,B4TYDEB6GKMZO031MB27,1,32,115000.0,360.0,2,2,2,1,116.0,25.0,2,10,2,5,-1,1,2
3,B4TYDEB6GKMZO031MB27,1,1,165000.0,360.0,2,2,2,1,87.0,25.0,1,9,2,-1,-1,1,1
4,B4TYDEB6GKMZO031MB27,1,31,145000.0,180.0,2,2,2,1,63.0,33.0,1,9,2,5,-1,1,2
5,B4TYDEB6GKMZO031MB27,1,1,135000.0,360.0,2,2,2,1,29.0,41.0,2,10,5,-1,-1,0,1
6,B4TYDEB6GKMZO031MB27,1,31,135000.0,180.0,2,2,2,1,93.0,33.0,1,9,2,5,-1,1,2
7,B4TYDEB6GKMZO031MB27,1,1,45000.0,360.0,2,2,2,1,16.0,25.0,3,10,5,-1,-1,1,1
8,B4TYDEB6GKMZO031MB27,1,1,195000.0,360.0,2,2,2,1,36.0,65.0,3,10,5,-1,-1,0,1
9,B4TYDEB6GKMZO031MB27,1,1,165000.0,360.0,2,2,2,1,36.0,33.0,2,10,5,-1,-1,1,1


In [18]:

# Models
models = {
    "RandomForestClassifier": {
            "model" :RandomForestClassifier(class_weight="balanced"),
            "params": {
                "n_estimators": [300,500,700],
                "min_samples_leaf": [5,7],
                "min_samples_split": [5,7],
                "max_depth": [15,25],
                "max_features": ["sqrt","log2"],
                "bootstrap": [True, False],
            }
        },
    "LogisticRegression": {
        "model":LogisticRegression(class_weight="balanced"),
        "params": {
            "max_iter": [1000,5000,10000,50000,100000],
            "C": [0.001,0.01,0.1,1,10,100,1000]
        }
    },
}


In [24]:

# Split (stratify so train/test keep the same approve/deny ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# roc_auc, not accuracy -- the target is ~70/30 imbalanced, and the product
# cares about ranking applicants by likelihood, not a raw 0.5-threshold label.
# verbose=4 prints each fit as it finishes, so a long run isn't a black box.
searchers = {
    "Grid": lambda pipeline, param_grid: GridSearchCV(
        pipeline, param_grid, cv=5, n_jobs=-1, scoring="roc_auc", verbose=4
    ),
    "Random": lambda pipeline, param_grid: RandomizedSearchCV(
        pipeline, param_grid, n_iter=30, cv=5, n_jobs=-1, scoring="roc_auc",
        verbose=4, random_state=42,
    ),
}

In [19]:

# Run -- each model gets both search strategies, so the comparison is direct.
result = {}
best_estimator = {}

for model_name, spec in tqdm(models.items(), desc="Training models"):
    pipeline = Pipeline([
        ("preprocess", build_preprocessor()),
        ("model", spec["model"]),
    ])
    param_grid = {f"model__{k}": v for k, v in spec["params"].items()}

    for search_name, make_search in searchers.items():
        key = f"{model_name} ({search_name})"
        print(f"\nTraining {key}")
        search = make_search(pipeline, param_grid)
        search.fit(X_train, y_train)
        result[key] = {
            "Best score": search.best_score_,
            "Best params": search.best_params_,
        }
        best_estimator[key] = search.best_estimator_
        print(f"Done: {key} | Best score: {search.best_score_:.4f}\n")

Training models:   0%|          | 0/2 [00:00<?, ?it/s]


Training RandomForestClassifier (Grid)
Fitting 5 folds for each of 96 candidates, totalling 480 fits
[CV 1/5] END model__bootstrap=True, model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=5, model__n_estimators=300;, score=0.914 total time= 1.1min
[CV 3/5] END model__bootstrap=True, model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=5, model__n_estimators=300;, score=0.913 total time= 1.1min
[CV 5/5] END model__bootstrap=True, model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=5, model__n_estimators=300;, score=0.912 total time= 1.1min
[CV 2/5] END model__bootstrap=True, model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=5, model__n_estimators=300;, score=0.917 total time= 1.1min
[CV 4/5] END model__bootstrap=True, model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=5, model__m

/Users/a70411/PycharmProjects/Let me get a LOAN/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV 2/5] END model__bootstrap=False, model__max_depth=25, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=5, model__n_estimators=500;, score=0.921 total time= 2.9min
[CV 1/5] END model__bootstrap=False, model__max_depth=25, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=7, model__n_estimators=300;, score=0.919 total time= 1.7min
[CV 3/5] END model__bootstrap=False, model__max_depth=25, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=5, model__n_estimators=500;, score=0.917 total time= 2.8min
[CV 4/5] END model__bootstrap=False, model__max_depth=25, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=5, model__n_estimators=500;, score=0.919 total time= 2.8min
[CV 5/5] END model__bootstrap=False, model__max_depth=25, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=5, model__n_estimators=500;, score=0.917 total time= 2.8min
[CV 2/5] END model__

/Users/a70411/PycharmProjects/Let me get a LOAN/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV 1/5] END model__bootstrap=True, model__max_depth=25, model__max_features=sqrt, model__min_samples_leaf=7, model__min_samples_split=7, model__n_estimators=300;, score=0.918 total time= 1.2min
[CV 2/5] END model__bootstrap=True, model__max_depth=25, model__max_features=sqrt, model__min_samples_leaf=7, model__min_samples_split=7, model__n_estimators=300;, score=0.920 total time= 1.2min
[CV 2/5] END model__bootstrap=False, model__max_depth=25, model__max_features=sqrt, model__min_samples_leaf=5, model__min_samples_split=5, model__n_estimators=500;, score=0.921 total time= 2.9min
[CV 3/5] END model__bootstrap=True, model__max_depth=25, model__max_features=sqrt, model__min_samples_leaf=7, model__min_samples_split=7, model__n_estimators=300;, score=0.916 total time= 1.2min
[CV 1/5] END model__bootstrap=False, model__max_depth=25, model__max_features=log2, model__min_samples_leaf=7, model__min_samples_split=7, model__n_estimators=500;, score=0.917 total time= 2.4min
[CV 3/5] END model__boo

Training models:  50%|█████     | 1/2 [2:04:32<2:04:32, 7472.24s/it]

Done: RandomForestClassifier (Random) | Best score: 0.9186


Training LogisticRegression (Grid)
Fitting 5 folds for each of 35 candidates, totalling 175 fits
[CV 2/5] END model__C=0.001, model__max_iter=1000;, score=0.858 total time=   1.3s
[CV 3/5] END model__C=0.001, model__max_iter=1000;, score=0.853 total time=   1.3s
[CV 5/5] END model__C=0.001, model__max_iter=1000;, score=0.853 total time=   1.4s
[CV 4/5] END model__C=0.001, model__max_iter=1000;, score=0.853 total time=   1.4s
[CV 2/5] END model__C=0.001, model__max_iter=5000;, score=0.858 total time=   1.4s
[CV 3/5] END model__C=0.001, model__max_iter=5000;, score=0.853 total time=   1.4s
[CV 1/5] END model__C=0.001, model__max_iter=5000;, score=0.853 total time=   1.4s
[CV 4/5] END model__C=0.001, model__max_iter=5000;, score=0.853 total time=   1.4s
[CV 1/5] END model__C=0.001, model__max_iter=1000;, score=0.853 total time=   1.5s
[CV 5/5] END model__C=0.001, model__max_iter=5000;, score=0.853 total time=   1.4s
[CV 1/5] END

Training models: 100%|██████████| 2/2 [2:05:33<00:00, 3766.59s/it]  

Done: LogisticRegression (Random) | Best score: 0.8621



In [20]:

# Evaluate
def best_model_cv():
    best_model_name = max(result, key=lambda x: result[x]["Best score"])
    best_model = best_estimator[best_model_name]
    test_accuracy = best_model.score(X_test, y_test)
    test_auc = roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])

    joblib.dump(best_model, MODEL_DIR / "best_model.joblib")

    print(f"\nBest model: {best_model_name}, Test accuracy: {test_accuracy:.4f}, Test ROC-AUC: {test_auc:.4f}")
    return [best_model_name, best_model, test_auc]

best_model = best_model_cv()
print(best_model)



Best model: RandomForestClassifier (Random), Test accuracy: 0.8352, Test ROC-AUC: 0.9180
['RandomForestClassifier (Random)', Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('ohe',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['lei', 'loan_type',
                                                   'loan_purpose',
                                                   'occupancy_type',
                                                   'applicant_credit_score_type',
                                                   'co-applicant_credit_score_type',
                                                   'interest_only_payment',
                                                   'negative_amortization',
       

In [21]:

y_pred = best_model[1].predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(best_model[2])

              precision    recall  f1-score   support

           0       0.69      0.82      0.75     18774
           1       0.92      0.84      0.88     44636

    accuracy                           0.84     63410
   macro avg       0.80      0.83      0.81     63410
weighted avg       0.85      0.84      0.84     63410

[[15404  3370]
 [ 7083 37553]]
0.9180310910073461


In [26]:
BANK_LEIS = {
    "Truist Bank": "JJKC32MCHWDI71265Z06",
    "Wells Fargo Bank, N.A.": "KB1H1DSPRFMYMCUFXT09",
    "Bank of America, N.A.": "B4TYDEB6GKMZO031MB27",
}

# Per-bank breakdown -- same metrics as above, sliced by lei instead of pooled.
y_proba = best_model[1].predict_proba(X_test)[:, 1]

for bank, lei in BANK_LEIS.items():
    mask = (X_test["lei"] == lei).to_numpy()
    print(f"=== {bank} (n={mask.sum()}) ===")
    print(classification_report(y_test[mask], y_pred[mask]))
    print(confusion_matrix(y_test[mask], y_pred[mask]))
    print(f"ROC-AUC: {roc_auc_score(y_test[mask], y_proba[mask]):.4f}")
    print(f"Actual approval rate:    {y_test[mask].mean():.1%}")
    print(f"Predicted approval rate: {y_pred[mask].mean():.1%}")
    print()

=== Truist Bank (n=26157) ===
              precision    recall  f1-score   support

           0       0.66      0.65      0.66      6655
           1       0.88      0.89      0.88     19502

    accuracy                           0.83     26157
   macro avg       0.77      0.77      0.77     26157
weighted avg       0.83      0.83      0.83     26157

[[ 4347  2308]
 [ 2196 17306]]
ROC-AUC: 0.8548
Actual approval rate:    74.6%
Predicted approval rate: 75.0%

=== Wells Fargo Bank, N.A. (n=21164) ===
              precision    recall  f1-score   support

           0       0.73      0.93      0.81      6116
           1       0.97      0.86      0.91     15048

    accuracy                           0.88     21164
   macro avg       0.85      0.89      0.86     21164
weighted avg       0.90      0.88      0.88     21164

[[ 5684   432]
 [ 2154 12894]]
ROC-AUC: 0.9638
Actual approval rate:    71.1%
Predicted approval rate: 63.0%

=== Bank of America, N.A. (n=16089) ===
              p

In [28]:
from sklearn.model_selection import cross_val_predict

# Honest, non-leaked probabilities on the training set (5-fold CV, refits only
# the already-chosen best model, not a hyperparameter search) -- X_test stays
# untouched so the final evaluation below is clean.
y_train_proba = cross_val_predict(
    best_model[1], X_train, y_train, cv=5, method="predict_proba", n_jobs=-1
)[:, 1]

# Per-bank threshold: the cutoff that makes predicted approval rate match each
# bank's actual (historical) approval rate, instead of one global 0.5 for all.
thresholds = {}
for bank, lei in BANK_LEIS.items():
    mask = (X_train["lei"] == lei).to_numpy()
    actual_rate = y_train[mask].mean()
    thresholds[bank] = pd.Series(y_train_proba[mask]).quantile(1 - actual_rate)
    print(f"{bank}: actual approval rate {actual_rate:.1%} -> threshold {thresholds[bank]:.3f}")

Truist Bank: actual approval rate 74.5% -> threshold 0.502
Wells Fargo Bank, N.A.: actual approval rate 71.0% -> threshold 0.333
Bank of America, N.A.: actual approval rate 62.8% -> threshold 0.353


In [29]:

# Apply the tuned thresholds to the untouched test set -- compare against the
# default-0.5 breakdown above.
for bank, lei in BANK_LEIS.items():
    mask = (X_test["lei"] == lei).to_numpy()
    y_pred_tuned = (y_proba[mask] >= thresholds[bank]).astype(int)
    print(f"=== {bank} (n={mask.sum()}, threshold={thresholds[bank]:.3f}) ===")
    print(classification_report(y_test[mask], y_pred_tuned))
    print(confusion_matrix(y_test[mask], y_pred_tuned))
    print(f"Actual approval rate:    {y_test[mask].mean():.1%}")
    print(f"Predicted approval rate: {y_pred_tuned.mean():.1%}")
    print()

=== Truist Bank (n=26157, threshold=0.502) ===
              precision    recall  f1-score   support

           0       0.66      0.66      0.66      6655
           1       0.88      0.89      0.88     19502

    accuracy                           0.83     26157
   macro avg       0.77      0.77      0.77     26157
weighted avg       0.83      0.83      0.83     26157

[[ 4361  2294]
 [ 2233 17269]]
Actual approval rate:    74.6%
Predicted approval rate: 74.8%

=== Wells Fargo Bank, N.A. (n=21164, threshold=0.333) ===
              precision    recall  f1-score   support

           0       0.82      0.82      0.82      6116
           1       0.93      0.93      0.93     15048

    accuracy                           0.90     21164
   macro avg       0.87      0.88      0.88     21164
weighted avg       0.90      0.90      0.90     21164

[[ 5045  1071]
 [ 1098 13950]]
Actual approval rate:    71.1%
Predicted approval rate: 71.0%

=== Bank of America, N.A. (n=16089, threshold=0.353) 

In [30]:

# Persist thresholds keyed by lei alongside the model, so predict.py can load both and apply the right cutoff.
thresholds_by_lei = {lei: thresholds[bank] for bank, lei in BANK_LEIS.items()}
joblib.dump(thresholds_by_lei, MODEL_DIR / "thresholds.joblib")
print(thresholds_by_lei)

{'JJKC32MCHWDI71265Z06': np.float64(0.5021773821641907), 'KB1H1DSPRFMYMCUFXT09': np.float64(0.3333525210956466), 'B4TYDEB6GKMZO031MB27': np.float64(0.3534488895676583)}


In [14]:
df[df['approved'] == 0].iloc[10000]

lei                               JJKC32MCHWDI71265Z06
loan_type                                            1
loan_purpose                                        32
loan_amount                                   205000.0
loan_term                                        360.0
negative_amortization                                2
interest_only_payment                                2
balloon_payment                                      2
occupancy_type                                       1
income                                            40.0
debt_to_income_ratio                              55.0
applicant_credit_score_type                          3
co-applicant_credit_score_type                      10
aus-1                                                2
aus-2                                               -1
aus-3                                               -1
approved                                             0
num_aus_used                                         1
Name: 3385